# Class imbalance on the role-aware line graph: entropy regularisation vs. oversampling

In [ ]:
# ---------------- settings ----------------
import os
TASK      = "one-way"                               # "one-way" or "two-way"
MODEL     = "GAT-SAGE"                              # "GAT-SAGE" (reference) or "GraphSAGE"
LABEL     = "threshold_0.66"                        # one-way only: "threshold_0.66" (reference) or "effect>0"
THRESHOLD = 0.66
LAMBDAS   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]   # ME balancing factors (baseline = 0)
RUN_OVERSAMPLING = True
EPOCHS    = int(os.environ.get("EPOCHS", 3000))
HIDDEN, HEADS, LR, SEED = 100, 4, 1e-3, 42
OUT_DIR   = "results_imbalance"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn as dglnn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

device = torch.device("cpu")
if torch.cuda.is_available():
    try:
        dgl.graph(([0], [0])).to("cuda")          # does this DGL build support the GPU?
        device = torch.device("cuda")
    except Exception:
        print("WARNING: PyTorch sees a GPU, but this DGL build is CPU-only -> running on CPU (slow).")
print("device:", device, "| torch", torch.__version__, "| dgl", dgl.__version__)

## 1. Data, label, features, folds

In [ ]:
if TASK == "one-way":
    df = pd.read_csv("Data_msystems/Features.csv")
    df = df[df["Carbon"] != "Water"].reset_index(drop=True)
    FEATURE_COLS = list(df.columns[5:18])
    effect = df["2 on 1: Effect"].values
    if LABEL == "effect>0":
        y = (effect > 0).astype(int)
    else:
        y = ((effect - effect.min()) / (effect.max() - effect.min()) >= THRESHOLD).astype(int)
    CLASSES = ["Negative", "Positive"]
    TAG = f"one-way_{LABEL}_{MODEL}"
else:
    df = pd.read_csv("Data_msystems/Features_two_way.csv")
    df = df[df["Carbon"] != "Water"].reset_index(drop=True)
    FEATURE_COLS = list(df.columns[6:19])
    pos_12, pos_21 = df["1 on 2: Effect"].values > 0, df["2 on 1: Effect"].values > 0
    y = np.where(pos_12 & pos_21, 0, np.where(~pos_12 & ~pos_21, 1, 2))      # 0 mutualism, 1 competition, 2 parasitism
    CLASSES = ["Mutualism", "Competition", "Parasitism"]
    TAG = f"two-way_{MODEL}"
K = len(CLASSES)
assert len(FEATURE_COLS) == 13 and not any("Effect" in c for c in FEATURE_COLS), FEATURE_COLS
X = MinMaxScaler().fit_transform(df[FEATURE_COLS].values)

# one group per co-culture (unordered species pair + carbon): both directions stay in the same fold
groups = df.apply(lambda r: "_".join(sorted([str(r["Bug 1"]), str(r["Bug 2"])])) + "_" + str(r["Carbon"]), axis=1).values
FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups=groups))
print(f"task {TAG} | nodes {len(df)} | class %:", {c: round(100 * np.mean(y == k), 1) for k, c in enumerate(CLASSES)})

## 2. Role-aware line graph
Two measurements are connected if, under the same carbon condition, they share the same effector (`Bug 2`) or the same affected species (`Bug 1`).

In [ ]:
src, dst = [], []
for role in ["Bug 2", "Bug 1"]:                     # same effector, then same affected
    for idx in df.groupby([role, "Carbon"]).indices.values():
        if len(idx) > 1:
            u, v = np.meshgrid(idx, idx)
            keep = u != v
            src.append(u[keep]); dst.append(v[keep])
EDGES = np.unique(np.stack([np.concatenate(src), np.concatenate(dst)], 1), axis=0)   # no duplicate edges
g = dgl.graph((torch.tensor(EDGES[:, 0]), torch.tensor(EDGES[:, 1])), num_nodes=len(df))
print("nodes:", g.num_nodes(), "| directed edges:", g.num_edges())

## 3. Model, the two imbalance strategies, and cross-validation

In [ ]:
class GAT_SAGE(nn.Module):                # GAT layer (4 heads) -> GraphSAGE layer
    def __init__(s, i, h, o, heads=HEADS):
        super().__init__()
        s.gat1 = dglnn.GATConv(i, h, num_heads=heads, activation=F.relu)
        s.conv2 = dglnn.SAGEConv(h * heads, o, aggregator_type="mean")
    def forward(s, g, x):
        return s.conv2(g, s.gat1(g, x).flatten(1))

class SAGE(nn.Module):                    # two GraphSAGE layers (mean aggregation)
    def __init__(s, i, h, o):
        super().__init__()
        s.conv1 = dglnn.SAGEConv(i, h, aggregator_type="mean")
        s.conv2 = dglnn.SAGEConv(h, o, aggregator_type="mean")
    def forward(s, g, x):
        return s.conv2(g, F.relu(s.conv1(g, x)))

BUILD = {"GraphSAGE": lambda: SAGE(13, HIDDEN, K), "GAT-SAGE": lambda: GAT_SAGE(13, HIDDEN, K)}

def me_loss(logits, target, lam):
    """Mixed entropy minimisation: CE + lam * mean predictive entropy (lam = 0 -> plain cross-entropy)."""
    ce = F.cross_entropy(logits, target)
    if lam == 0:
        return ce
    entropy = -(F.softmax(logits, 1) * F.log_softmax(logits, 1)).sum(1).mean()      # R(y_hat)
    return ce + lam * entropy

def oversample(tr, rng):
    """Training node indices with the smaller classes drawn again (with replacement) up to the size of the largest class."""
    cls, cnt = np.unique(y[tr], return_counts=True)
    extra = [rng.choice(tr[y[tr] == c], cnt.max() - n, replace=True) for c, n in zip(cls, cnt) if n < cnt.max()]
    return np.concatenate([tr] + extra)

def fold_metrics(true, prob):
    pred = prob.argmax(1)
    r = {"accuracy": accuracy_score(true, pred), "f1_macro": f1_score(true, pred, average="macro", zero_division=0)}
    if K == 2:
        r.update(precision=precision_score(true, pred, zero_division=0), recall=recall_score(true, pred, zero_division=0),
                 f1=f1_score(true, pred, zero_division=0), roc_auc=roc_auc_score(true, prob[:, 1]),
                 pr_auc=average_precision_score(true, prob[:, 1]))
    else:
        for fn_, nm in [(precision_score, "precision"), (recall_score, "recall"), (f1_score, "f1")]:
            for c, v in zip(CLASSES, fn_(true, pred, average=None, labels=list(range(K)), zero_division=0)):
                r[f"{nm}_{c}"] = v
    return r

def run(name, lam=0.0, oversampling=False):
    gd = g.to(device); xd = torch.tensor(X, dtype=torch.float32, device=device); yd = torch.tensor(y, device=device)
    rows, cm = [], np.zeros((K, K), dtype=int)
    for k, (tr, va) in enumerate(FOLDS, start=1):
        t0 = time.time(); torch.manual_seed(SEED + k)
        idx = oversample(tr, np.random.default_rng(SEED + k)) if oversampling else tr
        idx = torch.tensor(idx, device=device)
        model = BUILD[MODEL]().to(device); opt = torch.optim.Adam(model.parameters(), lr=LR)
        for ep in range(EPOCHS):
            model.train()
            loss = me_loss(model(gd, xd)[idx], yd[idx], lam)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            prob = F.softmax(model(gd, xd)[va], 1).cpu().numpy()          # VALIDATION nodes only
        rows.append({"method": name, "fold": k, **fold_metrics(y[va], prob)})
        cm += confusion_matrix(y[va], prob.argmax(1), labels=list(range(K)))
        print(f"  {name} fold {k}: F1 {rows[-1]['f1' if K == 2 else 'f1_macro']:.4f} | {time.time() - t0:.0f}s")
    return pd.DataFrame(rows), cm

## 4. Run the baseline, ME for every λ, and oversampling (same folds)

In [ ]:
CONFIGS = [("Baseline", 0.0, False)] + [(f"ME λ={l}", l, False) for l in LAMBDAS]
if RUN_OVERSAMPLING:
    CONFIGS.append(("Oversampling", 0.0, True))
RESULTS, CMS = {}, {}
for name, lam, ovs in CONFIGS:
    print(f"\n=== {name} ===")
    RESULTS[name], CMS[name] = run(name, lam, ovs)
    RESULTS[name].to_csv(f"{OUT_DIR}/folds_imbalance_{TAG}_{name.replace(' ', '_').replace('λ', 'lambda')}.csv", index=False)

## 5. Summary (mean ± std over the 5 folds)

In [ ]:
METRICS = (["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"] if K == 2 else
           ["accuracy", "f1_macro", "precision_Mutualism", "recall_Mutualism", "f1_Mutualism"])
MAIN = "f1" if K == 2 else "f1_macro"
fmt = lambda m, c: f"{RESULTS[m][c].mean():.4f} ± {RESULTS[m][c].std(ddof=0):.4f}"
summary = pd.DataFrame({m: [fmt(m, c) for c in METRICS] for m in RESULTS}, index=METRICS).T
summary.to_csv(f"{OUT_DIR}/imbalance_{TAG}_summary.csv")
display(summary)
ME_NAMES = [m for m in RESULTS if m.startswith("ME")]
BEST_ME = max(ME_NAMES, key=lambda m: RESULTS[m][MAIN].mean()) if ME_NAMES else None
print("best ME setting by mean", MAIN, ":", BEST_ME)

## 6. Tables in LaTeX (best value per column in bold)
(a) all balancing factors, (b) baseline vs. best ME vs. oversampling.

In [ ]:
names = {"accuracy": "Accuracy", "precision": "Precision", "recall": "Recall", "f1": "F1-score", "roc_auc": "ROC-AUC",
         "pr_auc": "PR-AUC", "f1_macro": "Macro F1", "precision_Mutualism": "Mut. precision",
         "recall_Mutualism": "Mut. recall", "f1_Mutualism": "Mut. F1"}
task_text = ("one-way interaction prediction (positive if the effect is $>0$)" if TASK == "one-way" and LABEL == "effect>0" else
             "one-way interaction prediction (positive if the min--max scaled effect is $\\geq 0.66$)" if TASK == "one-way" else
             "two-way interaction prediction")

def latex_table(rows, row_names, caption, label, first_col):
    best = {c: max(rows, key=lambda m: RESULTS[m][c].mean()) for c in METRICS}
    def cell(m, c):
        s = f"{RESULTS[m][c].mean():.4f} $\\pm$ {RESULTS[m][c].std(ddof=0):.4f}"
        return f"\\textbf{{{s}}}" if best[c] == m else s
    lines = [r"\begin{table}[t]", r"\centering", r"\small", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{l" + "c" * len(METRICS) + "}", r"\toprule",
             f"{first_col} & " + " & ".join(names[c] for c in METRICS) + r" \\", r"\midrule"]
    lines += [f"{row_names[m]} & " + " & ".join(cell(m, c) for c in METRICS) + r" \\" for m in rows]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

lam_rows = ["Baseline"] + ME_NAMES
tab_a = latex_table(lam_rows, {m: ("0.0" if m == "Baseline" else m.split("=")[1]) for m in lam_rows},
                    f"{MODEL} with mixed entropy minimisation for different balancing factors $\\lambda$, {task_text} "
                    r"(mean $\pm$ standard deviation, $n = 5$ folds; $\lambda = 0$ is plain cross-entropy). The best value in each column is shown in bold.",
                    f"tab:balancing_factors_{TASK}_{MODEL}", r"$\lambda$")
comp_rows = ["Baseline"] + ([BEST_ME] if BEST_ME else []) + (["Oversampling"] if "Oversampling" in RESULTS else [])
tab_b = latex_table(comp_rows, {"Baseline": f"{MODEL} (baseline)", "Oversampling": "Oversampling",
                                **({BEST_ME: f"Entropy regularisation ($\\lambda = {BEST_ME.split('=')[1]}$)"} if BEST_ME else {})},
                    f"Strategies against class imbalance for {MODEL}, {task_text} "
                    r"(mean $\pm$ standard deviation, $n = 5$ folds). The best value in each column is shown in bold.",
                    f"tab:imbalance_methods_{TASK}_{MODEL}", "Method")
for name, t in [("a", tab_a), ("b", tab_b)]:
    open(f"{OUT_DIR}/table_imbalance_{TAG}_{name}.tex", "w").write(t)
    print(t, "\n")

## 7. Paired t-tests against the baseline (same folds)

In [ ]:
from scipy.stats import ttest_rel
TEST = ["f1", "recall", "pr_auc"] if K == 2 else ["f1_macro", "recall_Mutualism", "f1_Mutualism"]
for m in [x for x in RESULTS if x != "Baseline"]:
    for c in TEST:
        t, p = ttest_rel(RESULTS[m][c], RESULTS["Baseline"][c])
        print(f"{m:14s} vs Baseline  {c:17s}: diff {RESULTS[m][c].mean() - RESULTS['Baseline'][c].mean():+.4f}  p = {p:.4f}")

## 8. Confusion matrices normalised by the total
All cells together sum to 100%; the number in brackets is the count summed over the 5 validation folds.

In [ ]:
import matplotlib.pyplot as plt
show = comp_rows
ticks = ["Negative", "Positive"] if K == 2 else ["Mut", "Comp", "Para"]
fig, axes = plt.subplots(1, len(show), figsize=(4.8 * len(show), 4.6))
for ax, m in zip(np.atleast_1d(axes), show):
    counts = CMS[m]; pct = counts / counts.sum() * 100                      # normalised by the total
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=pct.max())
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{pct[i, j]:.1f}%\n({counts[i, j]})", ha="center", va="center", fontsize=12,
                    color="white" if pct[i, j] > 0.6 * pct.max() else "black")
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels(ticks, fontsize=11); ax.set_yticklabels(ticks, fontsize=11)
    ax.set_title(m.replace("ME λ=", "Entropy reg. λ="), fontsize=14); ax.set_xlabel("Predicted", fontsize=12)
np.atleast_1d(axes)[0].set_ylabel("True", fontsize=12)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_imbalance_{TAG}.pdf", bbox_inches="tight"); plt.show()